In [0]:
import json, os, sys
from datetime import datetime, timezone
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common, refresh

# refresh/02_refresh_in: TARGET half of a controlled refresh (5.11). test: run by the refresh workflow as sp_<code>_test,
# followed by the normal test job (id maps, reconciliation, Gold, smoke). dev: run by a person in their own dev Git
# folder, then their own dev job. Adds the latest files in ops.refresh_in to the target's Silver (rows already there are
# skipped) and remakes id_fingerprint with the target's own key. A Silver rebuild from Bronze (silver_rebuild) removes them.
dbutils.widgets.dropdown("env", "test", ["test", "dev"])
env = dbutils.widgets.get("env")
cfg = common.load_config(env)
if not cfg["refresh_in"]:
    raise ValueError(f"{env} does not take refreshes (config/{env}.yml)")
rcfg, parents, ids = refresh.load()
base = f"/Volumes/{cfg['catalog']}/ops/refresh_in"
run = dbutils.fs.head(f"{base}/_latest").strip()
man = json.loads(dbutils.fs.head(f"{base}/{run}/_manifest.json"))
if man["to"] != env:
    raise ValueError(f"run {run} was made for {man['to']}, not {env}")
tables = [t for t in refresh.order(parents) if t in man["rows"]]
counts = refresh.load_in(spark, cfg["catalog"], f"{base}/{run}", common.match_key(env), rcfg, tables,
                         {t: parents[t][1] for t in tables})
diff = {t: (counts[t][0], man["rows"][t]) for t in tables if counts[t][0] != man["rows"][t]}
if diff:
    raise RuntimeError(f"files differ from the manifest (rows read, rows written): {diff}")
for f in dbutils.fs.ls(base):                                         # keep only the run just loaded
    if f.name.rstrip("/") not in (run, "_latest"):
        dbutils.fs.rm(f.path, True)
for t in tables:
    print(f"{t:22} {counts[t][0]:>8} rows, {counts[t][1]:>8} added")
added = sum(a for _, a in counts.values())
common.log_run(spark, cfg, "refresh_in", "success", added, f"from {man['from']}, run {run}, {man['pct']}% of people")
dbutils.notebook.exit(json.dumps({"run": run, "added": added}))